# Baseline, Null Spasial, Bootstrap, dan Evaluasi Validitas


In [ ]:
from pathlib import Path
import json, warnings, re
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (adjusted_rand_score as ARI, normalized_mutual_info_score as NMI,
                             cohen_kappa_score)
from scipy.sparse import csr_matrix
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

SEED = 42
SEEDS = list(range(10))         
RES_GRID = [round(x, 2) for x in np.arange(0.5, 2.51, 0.05)]   
LSA_DIM = 100                   
B_PERM = 9999                   
N_NULL_SPATIAL = 999            
B_BOOT = 500                   
PROBE_N = 12000                 
N_PER_TOPIC = 20                
BASE_DIR = Path.cwd()
SEMANTIC_DIR = BASE_DIR / "semantic_output"
CLEANING_DIR = BASE_DIR / "cleaning_output"
OUT_DIR = SEMANTIC_DIR / "validation_06"; OUT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_DIRS = [SEMANTIC_DIR, BASE_DIR, CLEANING_DIR, BASE_DIR.parent / "semantic_output", BASE_DIR.parent / "cleaning_output"]

def find_file(pattern):
    for d in SEARCH_DIRS:
        hits = sorted(Path(d).glob(pattern)) if Path(d).exists() else []
        if hits:
            return hits[0]
    return None

PATHS = {
    "nodes":      find_file("node_table_kelurahan_tahun.csv"),
    "membership": find_file("community_membership_final.csv"),
    "config":     find_file("config_final.json"),
    "data":       find_file("data_cleaned.csv"),
    "raw_mean":   find_file("complaint_embeddings_raw_mean*.npy"),
    "raw_cls":    find_file("complaint_embeddings_raw_cls*.npy"),     
    "meta":       find_file("complaint_embedding_metadata.csv"),
}
for k, v in PATHS.items():
    print(f"{k:11s}: {v}")
for k in ["nodes", "membership", "config", "data", "raw_mean", "meta"]:
    assert PATHS[k] is not None, f"File '{k}' tidak ditemukan; letakkan di salah satu folder: {[str(d) for d in SEARCH_DIRS]}"

RESULTS = {}   

## 1. Muat data dan validasi keselarasan

Pemeriksaan penting: 
- (i) urutan tiket pada metadata embedding = urutan `data_cleaned.csv`; 
- (ii) embedding node dapat **direproduksi** dari embedding tiket (centering pada level tiket, L2, rata-rata per node, L2); 
- (iii) partisi Louvain pada graf yang direkonstruksi sama dengan `community_membership_final.csv`.

In [ ]:
nodes = pd.read_csv(PATHS["nodes"], low_memory=False).sort_values("node_id").reset_index(drop=True)
mem = pd.read_csv(PATHS["membership"], low_memory=False).sort_values("node_id").reset_index(drop=True)
cfg = json.load(open(PATHS["config"]))
df = pd.read_csv(PATHS["data"], low_memory=False)
meta = pd.read_csv(PATHS["meta"], low_memory=False)
for t in (nodes, mem, df):
    t["lokasi_key"] = t["lokasi_key"].astype(str)
    t["tahun"] = pd.to_numeric(t["tahun"]).astype(int)

N = len(nodes)
assert np.array_equal(nodes["node_id"].to_numpy(), np.arange(N)) and np.array_equal(mem["node_id"].to_numpy(), np.arange(N))
assert (nodes["lokasi_key"].to_numpy() == mem["lokasi_key"].to_numpy()).all() and (nodes["tahun"].to_numpy() == mem["tahun"].to_numpy()).all()
nodes["komunitas"] = mem["komunitas"].astype(int).to_numpy()
lab_main = nodes["komunitas"].to_numpy(); K_MAIN = int(nodes["komunitas"].nunique())
lk = nodes["lokasi_key"].to_numpy(); yr = nodes["tahun"].to_numpy()
same_yr = yr[:, None] == yr[None, :]
iu = np.triu_indices(N, 1)
print("Konfigurasi dari Notebook 04:", cfg, "| node:", N, "| komunitas:", K_MAIN)

# tiket -> node
assert len(meta) == len(df) and (meta["no_tiket"].to_numpy() == df["no_tiket"].to_numpy()).all(), "Urutan metadata embedding berbeda dari data_cleaned."
nid = df.merge(nodes[["lokasi_key", "tahun", "node_id"]], on=["lokasi_key", "tahun"], how="left")["node_id"]
assert nid.notna().all(); nid = nid.astype(int).to_numpy()
assert (np.bincount(nid, minlength=N) == nodes["n_pengaduan"].to_numpy()).all(), "Jumlah tiket per node tidak cocok dengan n_pengaduan."
dfm = df.assign(node_id=nid, komunitas=lab_main[nid])

def l2(a):
    return a / np.linalg.norm(a, axis=1, keepdims=True)

raw = np.load(PATHS["raw_mean"]).astype(np.float32)
assert raw.shape[0] == len(df)
Ec = l2(raw - raw.mean(0))
X = np.load(SEMANTIC_DIR / "node_embeddings_mean_centered.npy") if (SEMANTIC_DIR / "node_embeddings_mean_centered.npy").exists() else np.load(find_file("node_embeddings_mean_centered.npy"))
S_sum = np.zeros((N, Ec.shape[1])); np.add.at(S_sum, nid, Ec)
X_rebuilt = l2(S_sum / np.bincount(nid)[:, None])
cos_rebuilt = (X_rebuilt * X).sum(1)
print(f"Reproduksi embedding node dari embedding tiket: cosine minimum {cos_rebuilt.min():.6f}")
assert cos_rebuilt.min() > 0.9999, "Embedding node tidak dapat direproduksi dari embedding tiket; bootstrap tidak valid."

SIM = X @ X.T

# jarak
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2.0) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
coords = nodes[["lat_kelurahan", "lon_kelurahan"]].to_numpy(float)
D = haversine_km(coords[:, 0][:, None], coords[:, 1][:, None], coords[:, 0][None, :], coords[:, 1][None, :])
ukel = sorted(set(lk)); kidx = {k: i for i, k in enumerate(ukel)}; ki = np.array([kidx[k] for k in lk])
first_idx = [int(np.flatnonzero(lk == k)[0]) for k in ukel]
Dk = D[np.ix_(first_idx, first_idx)]

# profil topik per node (jumlah dan proporsi)
topics = sorted(df["kelompok_topik"].unique())
CNT = (dfm.groupby(["node_id", "kelompok_topik"]).size().unstack(fill_value=0)
          .reindex(index=np.arange(N), columns=topics, fill_value=0).to_numpy(float))
TP = CNT / CNT.sum(1, keepdims=True)
tpc = TP - TP.mean(0); assert (np.linalg.norm(tpc, axis=1) > 0).all(); tpc = l2(tpc)
ST = tpc @ tpc.T                                    # kemiripan profil topik (centered)
print("Topik:", len(topics))

In [ ]:
def geo_neighbors(K):
    M_ = Dk.copy(); np.fill_diagonal(M_, np.inf)
    idx = np.argsort(M_, axis=1, kind="stable")[:, :K]
    B = np.zeros(M_.shape, bool); B[np.arange(len(M_))[:, None], idx] = True
    B = B | B.T; np.fill_diagonal(B, True)
    return B

def knn_sym(M_, k, allowed):
    M_ = np.where(allowed, M_, -np.inf); np.fill_diagonal(M_, -np.inf)
    idx = np.argsort(-M_, axis=1, kind="stable")[:, :k]
    B = np.zeros(M_.shape, bool); B[np.arange(len(M_))[:, None], idx] = True
    return B | B.T

def candidate_matrix(scope, K):
    A = geo_neighbors(K)[np.ix_(ki, ki)].copy()
    if scope == "same_year":
        A &= same_yr
    elif scope == "spatio_temporal_consecutive":
        A &= np.abs(yr[:, None] - yr[None, :]) <= 1
    elif scope != "spatio_temporal":
        raise NotImplementedError(f"scope '{scope}' belum didukung di notebook ini (same_year+temporal memerlukan edge temporal eksplisit).")
    np.fill_diagonal(A, False)
    return A

def graph_from_similarity(Sim, A, M_):
    Wm = np.where(A, np.clip(Sim, 0, None), 0.0)
    keep = knn_sym(Wm, M_, Wm > 0) & (Wm > 0)
    return np.where(keep, Wm, 0.0)

def louvain_labels(Wm, seed, resolution=1.0):
    Gw = nx.from_numpy_array(Wm)
    parts = nx.community.louvain_communities(Gw, weight="weight", seed=seed, resolution=resolution)
    lab = np.zeros(len(Wm), int)
    for c, p in enumerate(parts):
        for n_ in p:
            lab[n_] = c
    return lab, Gw, parts

A_main = candidate_matrix(cfg["scope"], cfg["k_geo"])
W_main = graph_from_similarity(SIM, A_main, cfg["m"])
cand = [louvain_labels(W_main, s) for s in SEEDS]
qs = [nx.community.modularity(g_, p_, weight="weight") for _, g_, p_ in cand]
lab_rebuilt = cand[int(np.argmax(qs))][0]
ari_chk = ARI(lab_main, lab_rebuilt)
print(f"Partisi rekonstruksi vs community_membership_final.csv: ARI = {ari_chk:.4f} | sisi graf: {int((W_main > 0).sum() / 2)}")
assert ari_chk > 0.95, "Graf/partisi rekonstruksi tidak cocok dengan Notebook 04 (periksa config_final.json dan embedding)."

## A. Validasi representasi (RQ1)

Empat varian embedding node dibandingkan dengan profil topik yang **tidak ikut membentuk** embedding:
* Spearman antara kemiripan embedding dan kemiripan profil topik. Pasangan yang adil: embedding *raw* dengan profil topik mentah, embedding *centered* dengan profil topik centered.
* kNN (k=5) pada embedding: rata-rata kemiripan topik (centered) para tetangga, tanpa kelurahan yang sama.
* Uji level tiket: klasifikasi `kelompok_topik` dengan kNN cosine (subsampel, 5-fold).

In [ ]:
variants_node = {}
for nm in ["cls_raw", "cls_centered", "mean_raw", "mean_centered"]:
    f = find_file(f"node_embeddings_{nm}.npy")
    if f is not None:
        variants_node[nm] = np.load(f)
print("Varian node tersedia:", list(variants_node))

tp_raw_n = l2(TP)
S_topic_raw, S_topic_c = (tp_raw_n @ tp_raw_n.T), ST
same_kel = lk[:, None] == lk[None, :]
rows = []
for nm, Xv in variants_node.items():
    Sv = Xv @ Xv.T
    Sm = np.where(same_kel, -np.inf, Sv); np.fill_diagonal(Sm, -np.inf)
    nb = np.argsort(-Sm, axis=1, kind="stable")[:, :5]
    knn_topic = float(np.mean([ST[i, nb[i]].mean() for i in range(N)]))
    same_b = (same_kel & ~same_yr)
    rows.append({"representasi": nm,
                 "cos_rata2": float(Sv[iu].mean()), "cos_std": float(Sv[iu].std()),
                 "gap_kelurahan_sama": float(Sv[same_b].mean() - Sv[iu].mean()),
                 "spearman_vs_topik_mentah": spearmanr(Sv[iu], S_topic_raw[iu])[0],
                 "spearman_vs_topik_centered": spearmanr(Sv[iu], S_topic_c[iu])[0],
                 "knn5_kemiripan_topik": knn_topic})
repr_node = pd.DataFrame(rows).round(4)
repr_node.to_csv(OUT_DIR / "A_representation_node_level.csv", index=False, encoding="utf-8-sig")
display(repr_node)

In [ ]:
y_all = df["kelompok_topik"].to_numpy()
rng = np.random.default_rng(SEED)
sub = rng.choice(len(df), min(PROBE_N, len(df)), replace=False)
ys = pd.factorize(y_all[sub])[0]
groups_sub = nid[sub]
ticket_variants = {"mean_raw": l2(raw), "mean_centered": Ec}
if PATHS["raw_cls"] is not None:
    cls_raw = np.load(PATHS["raw_cls"]).astype(np.float32)
    ticket_variants.update({"cls_raw": l2(cls_raw), "cls_centered": l2(cls_raw - cls_raw.mean(0))})
else:
    print("Embedding CLS tingkat tiket tidak ditemukan; uji tiket hanya untuk varian mean.")

def knn_probe(Xs, y, groups, k=10, folds=5):
    from sklearn.metrics import f1_score
    accs, f1s = [], []
    splitter = GroupKFold(n_splits=folds)
    for tr, te in splitter.split(Xs, y, groups):
        Sx = Xs[te] @ Xs[tr].T
        kk = min(k, len(tr))
        nb = np.argpartition(-Sx, kk - 1, axis=1)[:, :kk]
        votes = np.array([np.bincount(y[tr][r], minlength=y.max() + 1).argmax() for r in nb])
        accs.append((votes == y[te]).mean())
        f1s.append(f1_score(y[te], votes, average="macro",
                            labels=np.arange(y.max() + 1), zero_division=0))
    return float(np.mean(accs)), float(np.mean(f1s))

rows = []
for nm, Xt in ticket_variants.items():
    acc, f1m = knn_probe(Xt[sub], ys, groups_sub)
    rows.append({"representasi": nm, "kNN_akurasi_groupCV": acc, "kNN_macroF1_groupCV": f1m})
probe = pd.DataFrame(rows).round(3)
base_rate = float(pd.Series(y_all).value_counts(normalize=True).iloc[0])
probe.to_csv(OUT_DIR / "A_representation_ticket_probe.csv", index=False, encoding="utf-8-sig")
display(probe)
print(f"Pembanding kelas terbesar (seluruh data): {base_rate:.1%}")
print("Hasil ini mengukur kesesuaian embedding dengan weak labels, bukan ground truth independen.")
RESULTS["representasi_node"] = repr_node.to_dict("records")
RESULTS["representasi_tiket"] = probe.to_dict("records")

## B. Pembanding pada jumlah kelompok yang sama (RQ3)



In [ ]:
# Representasi teks pembanding: TF-IDF -> LSA pada level node (teks_embedding = teks termasking)
def prep(t):
    t = re.sub(r"\[[A-Z_]+\]", " ", str(t).lower())
    return re.sub(r"\s+", " ", t)
docs = dfm.groupby("node_id")["teks_embedding"].apply(lambda s: " ".join(map(prep, s.fillna("")))).reindex(np.arange(N))
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=3, max_df=0.5, token_pattern=r"(?u)\b[a-zA-Z]{3,}\b")
Xtf = tfidf.fit_transform(docs.tolist())
Z = TruncatedSVD(n_components=LSA_DIM, random_state=SEED).fit_transform(Xtf)
Z = l2(l2(Z) - l2(Z).mean(0))
SIM_LSA = Z @ Z.T
print(f"TF-IDF: {Xtf.shape[1]} fitur | LSA: {LSA_DIM} dimensi | cosine rata-rata antar node {SIM_LSA[iu].mean():.3f}")
print("Korelasi kemiripan IndoBERT vs LSA (Spearman):", round(spearmanr(SIM[iu], SIM_LSA[iu])[0], 3))

SIMS = {"indobert": SIM[iu], "lsa": SIM_LSA[iu], "topik": ST[iu]}
D_iu, same_yr_iu = D[iu], same_yr[iu]
D_mean_same_year = float(D_iu[same_yr_iu].mean())
def mean_or_nan(v):
    return float(v.mean()) if v.size else float("nan")

def group_metrics(lab):
    lab = pd.factorize(np.asarray(lab))[0]
    m_all = lab[iu[0]] == lab[iu[1]]; sy = m_all & same_yr_iu; sz = np.bincount(lab)
    out = {"n_kelompok": len(sz), "ukuran_min": int(sz.min()), "ukuran_maks": int(sz.max())}
    for nm, v in SIMS.items():
        out[f"gain_{nm}"] = mean_or_nan(v[m_all]) - float(v.mean())
    out["spat_ratio"] = mean_or_nan(D_iu[sy]) / D_mean_same_year
    return out

### B. Pembanding representasi: sentence-transformer multilingual


In [ ]:

SBERT_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
SBERT_MAX_LEN = 256
SBERT_BATCH = 64
SBERT_CACHE = OUT_DIR / f"ticket_embeddings_{SBERT_MODEL.split('/')[-1]}.npy"

texts_all = df["teks_embedding"].fillna("").astype(str).tolist()
if SBERT_CACHE.exists():
    E_sb = np.load(SBERT_CACHE)
    assert E_sb.shape[0] == len(texts_all), "Cache tidak cocok dengan data; hapus file cache lalu jalankan ulang."
    print("Memuat cache embedding:", SBERT_CACHE.name)
else:
    from sentence_transformers import SentenceTransformer
    sb_model = SentenceTransformer(SBERT_MODEL)
    sb_model.max_seq_length = SBERT_MAX_LEN
    E_sb = sb_model.encode(texts_all, batch_size=SBERT_BATCH, show_progress_bar=True,
                           convert_to_numpy=True, normalize_embeddings=False).astype(np.float32)
    np.save(SBERT_CACHE, E_sb)

Ec_sb = l2(E_sb - E_sb.mean(0))
S_sb = np.zeros((N, Ec_sb.shape[1])); np.add.at(S_sb, nid, Ec_sb)
X_sb = l2(S_sb / np.bincount(nid)[:, None])
SIM_SB = X_sb @ X_sb.T
SIMS["sbert"] = SIM_SB[iu]            # menambah kolom gain_sbert di group_metrics

# Kualitas representasi terhadap kemiripan profil topik (tidak ikut membentuk embedding)
rep_quality = {
    "indobert_vs_topik": float(spearmanr(SIM[iu], ST[iu])[0]),
    "lsa_vs_topik":      float(spearmanr(SIM_LSA[iu], ST[iu])[0]),
    "sbert_vs_topik":    float(spearmanr(SIM_SB[iu], ST[iu])[0]),
    "indobert_vs_sbert": float(spearmanr(SIM[iu], SIM_SB[iu])[0]),
}
print("Spearman kemiripan node vs profil topik / antar-representasi:")
for k_, v_ in rep_quality.items():
    print(f"  {k_:20s} {v_:.3f}")
RESULTS["representasi_sbert"] = {"model": SBERT_MODEL, **{k_: round(v_, 4) for k_, v_ in rep_quality.items()}}

# Graf Louvain dengan bobot SBERT, dimasukkan ke dictionary `graphs`
EXTRA_GRAPHS = {
    "Louvain, bobot sentence-transformer multilingual (pembanding representasi)":
        graph_from_similarity(SIM_SB, A_main, cfg["m"])
}

In [ ]:
K_COMPARE = sorted({max(2, K_MAIN - 2), K_MAIN, K_MAIN + 2})
kec_codes = pd.factorize(nodes["kecamatan"])[0]

# Proyeksi koordinat ke meter (UTM zona 49S untuk Surabaya).
try:
    from pyproj import Transformer
    transformer = Transformer.from_crs(
        "EPSG:4326", "EPSG:32749", always_xy=True
    )
    xx, yy = transformer.transform(coords[:, 1], coords[:, 0])
    coords_m = np.column_stack([xx, yy])

    kec_xy = (
        pd.DataFrame({
            "x_m": xx,
            "y_m": yy,
            "kecamatan": nodes["kecamatan"]
        })
        .groupby("kecamatan")[["x_m", "y_m"]]
        .mean()
    )
    print("Baseline koordinat memakai UTM 49S (EPSG:32749), satuan meter.")

except ImportError:
    warnings.warn(
        "pyproj tidak tersedia; baseline koordinat memakai derajat "
        "geografis dan bersifat sementara."
    )
    coords_m = coords.copy()
    kec_xy = (
        nodes.groupby("kecamatan")[["lat_kelurahan", "lon_kelurahan"]]
        .mean()
    )

# KNN centroid kecamatan bukan adjacency polygon administratif/jaringan jalan.
kec_conn = kneighbors_graph(
    kec_xy.to_numpy(),
    n_neighbors=min(4, len(kec_xy) - 1),
    include_self=False
)

node_kec_pos = nodes["kecamatan"].map(
    {k: i for i, k in enumerate(kec_xy.index)}
).to_numpy()

runs = []

for k in K_COMPARE:
    g_ = AgglomerativeClustering(
        n_clusters=k,
        linkage="ward",
        connectivity=kec_conn
    ).fit_predict(kec_xy.to_numpy())

    runs.append((
        "Ward centroid kecamatan + konektivitas kNN "
        "(bukan kontiguitas polygon)",
        k,
        g_[node_kec_pos]
    ))

    runs.append((
        "Ward embedding IndoBERT + konektivitas kandidat geografis-temporal",
        k,
        AgglomerativeClustering(
            n_clusters=k,
            linkage="ward",
            connectivity=csr_matrix(A_main)
        ).fit_predict(X)
    ))

    for s in SEEDS:
        runs.append((
            "K-means koordinat terproyeksi saja (tanpa semantik)",
            k,
            KMeans(
                n_clusters=k,
                n_init=10,
                random_state=s
            ).fit_predict(coords_m)
        ))

        runs.append((
            "K-means embedding IndoBERT saja (tanpa lokasi)",
            k,
            KMeans(
                n_clusters=k,
                n_init=10,
                random_state=s
            ).fit_predict(X)
        ))

        runs.append((
            "K-means LSA/TF-IDF saja (tanpa lokasi)",
            k,
            KMeans(
                n_clusters=k,
                n_init=10,
                random_state=s
            ).fit_predict(Z)
        ))

graphs = {
    "Louvain, bobot IndoBERT (graf utama)": W_main,
    "Louvain, bobot LSA/TF-IDF (pembanding teks)":
        graph_from_similarity(SIM_LSA, A_main, cfg["m"]),
    "Louvain, bobot profil topik (berbasis label; bukan bukti independen)":
        graph_from_similarity(ST, A_main, cfg["m"]),
    **EXTRA_GRAPHS
}

pool = []

for nm, Wm in graphs.items():
    for res in RES_GRID:
        for s in SEEDS:
            lab_, _, _ = louvain_labels(Wm, s, res)
            pool.append((nm, len(np.unique(lab_)), lab_))

for k in K_COMPARE:
    runs += [
        (nm, k, lab_)
        for nm, kk, lab_ in pool
        if kk == k
    ]

per_run = pd.DataFrame([
    dict(metode=m, k_target=k, **group_metrics(l))
    for m, k, l in runs
])

gcols = ["gain_indobert", "gain_lsa", "gain_sbert", "gain_topik", "spat_ratio"]

agg = (
    per_run.groupby(["k_target", "metode"])
    .agg(
        n_run=("spat_ratio", "size"),
        n_kelompok=("n_kelompok", "median"),
        ukuran_min=("ukuran_min", "median"),
        **{c: (c, "mean") for c in gcols},
        **{c + "_sd": (c, "std") for c in gcols}
    )
    .reset_index()
)

agg.to_csv(
    OUT_DIR / "B_baselines_matched_k.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 75)

print("Gunakan k_target =", K_MAIN, "sebagai baris utama.")
print(
    "ARI/NMI terhadap partisi utama adalah kesesuaian antar-partisi, "
    "bukan akurasi ground truth."
)
display(agg.round(3))
print(agg[agg.k_target == K_MAIN][["metode", "n_run"]].to_string(index=False))   # target: n_run >= 10 untuk tiap metode Louvain/K-means

ref = pd.DataFrame([
    dict(
        metode="Partisi final (metode utama)",
        **group_metrics(lab_main)
    ),
    dict(
        metode="Kecamatan asli (jumlah kelompok tidak disamakan)",
        **group_metrics(kec_codes)
    )
])

display(ref.round(3))

ref.to_csv(
    OUT_DIR / "B_reference_rows.csv",
    index=False,
    encoding="utf-8-sig"
)

ct = pd.crosstab(nodes["kecamatan"], nodes["komunitas"])
purity = float(ct.max(axis=1).sum() / ct.values.sum())

print(
    f"Kesesuaian komunitas dengan kecamatan: "
    f"NMI = {NMI(kec_codes, lab_main):.3f} | "
    f"ARI = {ARI(kec_codes, lab_main):.3f} | "
    f"purity = {purity:.1%}"
)

print(
    "Modularitas dari graf berbeda tidak dibandingkan "
    "sebagai ranking kualitas langsung."
)

RESULTS["baseline_matched_k_utama"] = (
    agg[agg.k_target == K_MAIN]
    .round(4)
    .to_dict("records")
)

# PERBAIKAN: jangan membulatkan Series yang berisi string.
# Bulatkan hanya nilai numerik; pertahankan kolom teks apa adanya.
partisi_final_dict = ref.iloc[0].to_dict()

partisi_final_dict = {
    key: (
        round(float(value), 4)
        if isinstance(value, (int, float, np.integer, np.floating))
        and not pd.isna(value)
        else value
    )
    for key, value in partisi_final_dict.items()
}

RESULTS["partisi_final"] = partisi_final_dict

RESULTS["kesesuaian_kecamatan"] = {
    "NMI": float(NMI(kec_codes, lab_main)),
    "ARI": float(ARI(kec_codes, lab_main)),
    "purity": purity
}

print("Baseline comparison selesai.")
print("Partisi final:", RESULTS["partisi_final"])

In [ ]:
sub_k = agg[agg.k_target == K_MAIN]
fig, axes = plt.subplots(1, 4, figsize=(21, 4.8), sharex=True)
for ax, col, ttl in zip(axes, ["gain_topik", "gain_indobert", "gain_lsa", "gain_sbert"],
                        ["kohesi topik", "kohesi IndoBERT", "kohesi LSA/TF-IDF", "kohesi sentence-transformer"]):
    for _, r in sub_k.iterrows():
        ax.errorbar(r.spat_ratio, r[col], xerr=0 if np.isnan(r.spat_ratio_sd) else r.spat_ratio_sd,
                    yerr=0 if np.isnan(r[col + "_sd"]) else r[col + "_sd"], fmt="o", ms=7, capsize=3, label=r.metode)
    ax.scatter(ref.iloc[0].spat_ratio, ref.iloc[0][col], marker="*", s=240, color="k", zorder=5, label="Partisi final")
    ax.set_xlabel("spat_ratio (kecil = kompak)"); ax.set_ylabel(col); ax.set_title(ttl); ax.grid(alpha=0.25)
axes[0].legend(fontsize=6.5, loc="best")
fig.suptitle(f"Trade-off kohesi vs kekompakan geografis (jumlah kelompok = {K_MAIN})")
plt.tight_layout(); plt.savefig(OUT_DIR / "B_tradeoff.png", dpi=150); plt.show()

## C. Uji permutasi komposisi topik antar komunitas


In [ ]:
rng = np.random.default_rng(SEED)
comms = np.sort(np.unique(lab_main)); Lmat = (lab_main[:, None] == comms[None, :]).astype(float)

def _kl(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(a > 0, a * np.log2(a / b), 0.0)
    return r.sum(axis=-1)
def jsd_to_global(P, q):
    Q = np.broadcast_to(q, P.shape); Mx = 0.5 * (P + Q)
    return 0.5 * _kl(P, Mx) + 0.5 * _kl(Q, Mx)
def community_stats(Cm, Lm):
    G_ = Lm.T @ Cm; n_c = G_.sum(axis=1)
    d = jsd_to_global(G_ / n_c[:, None], Cm.sum(axis=0) / Cm.sum())
    return d, float((n_c / n_c.sum()) @ d)

d_obs, T_obs = community_stats(CNT, Lmat)
strata = [g_.index.to_numpy() for _, g_ in nodes.reset_index(drop=True).groupby(["kecamatan", "tahun"])]
null = {k: {"T": np.zeros(B_PERM), "d": np.zeros((B_PERM, len(comms)))} for k in ("N1", "N2")}
for b in range(B_PERM):
    d1, T1 = community_stats(CNT, Lmat[rng.permutation(N)]); null["N1"]["d"][b], null["N1"]["T"][b] = d1, T1
    # Acak proporsi topik antar-node dalam kecamatan x tahun, lalu
    # kalikan kembali dengan total tiket asli tiap node (volume tetap).
    TP_perm = TP.copy()
    for g_ in strata:
        if len(g_) > 1:
            TP_perm[g_] = TP[rng.permutation(g_)]
    Cp = TP_perm * CNT.sum(axis=1, keepdims=True)
    d2, T2 = community_stats(Cp, Lmat)
    null["N2"]["d"][b], null["N2"]["T"][b] = d2, T2

def pval(obs, nul):
    return (1 + np.sum(nul >= obs, axis=0)) / (len(nul) + 1)
def bh(p):
    p = np.asarray(p, float); n_ = len(p); o = np.argsort(p); q = np.empty(n_)
    q[o] = np.minimum.accumulate((p[o] * n_ / (np.arange(n_) + 1))[::-1])[::-1]
    return np.clip(q, 0, 1)

perm_summary = pd.DataFrame([{"hipotesis_nol": {"N1": "N1: label acak antar node", "N2": "N2: proporsi topik diacak dalam kecamatan x tahun; total tiket/node dipertahankan"}[nm],
    "T_teramati": T_obs, "T_nol_rata2": null[nm]["T"].mean(), "T_nol_sd": null[nm]["T"].std(ddof=1),
    "z": (T_obs - null[nm]["T"].mean()) / null[nm]["T"].std(ddof=1), "p_permutasi": float(pval(T_obs, null[nm]["T"]))} for nm in ("N1", "N2")])
perm_comm = pd.DataFrame({"komunitas": comms, "n_node": Lmat.sum(0).astype(int), "n_pengaduan": (Lmat.T @ CNT).sum(1).astype(int),
                          "JSD_ke_global": d_obs, "JSD_nol_N2_rata2": null["N2"]["d"].mean(0),
                          "p_N1": pval(d_obs, null["N1"]["d"]), "p_N2": pval(d_obs, null["N2"]["d"])})
perm_comm["q_BH_N1"] = bh(perm_comm.p_N1); perm_comm["q_BH_N2"] = bh(perm_comm.p_N2)
display(perm_summary.round(4)); display(perm_comm.round(4))
perm_summary.to_csv(OUT_DIR / "C_permutation_summary.csv", index=False, encoding="utf-8-sig")
perm_comm.to_csv(OUT_DIR / "C_permutation_per_community.csv", index=False, encoding="utf-8-sig")
print(f"B = {B_PERM}; p terkecil yang mungkin = {1/(B_PERM+1):.4f}. Komunitas dengan q_BH_N2 >= 0,05 tidak terbukti berbeda melebihi pola tingkat kecamatan.")
RESULTS["permutasi"] = perm_summary.round(5).to_dict("records")
RESULTS["permutasi_komunitas_tak_signifikan_N2"] = [int(c) for c in perm_comm.loc[perm_comm.q_BH_N2 >= 0.05, "komunitas"]]

## D. Null spasial tambahan: partisi Voronoi geografis acak


In [ ]:
rng = np.random.default_rng(SEED)
def random_voronoi(k):
    seeds_ = rng.choice(len(ukel), size=k, replace=False)
    return np.argmin(Dk[:, seeds_], axis=1)[ki]

obs = group_metrics(lab_main)
null_rows = [group_metrics(random_voronoi(K_MAIN)) for _ in range(N_NULL_SPATIAL)]
null_df = pd.DataFrame(null_rows)
rows = []
for c in ["gain_indobert", "gain_lsa", "gain_topik", "spat_ratio"]:
    p = (1 + (null_df[c] >= obs[c]).sum()) / (len(null_df) + 1)
    rows.append({"metrik": c, "teramati": obs[c], "null_rata2": null_df[c].mean(), "null_sd": null_df[c].std(),
                 "null_p95": null_df[c].quantile(0.95), "p_satu_sisi(null>=teramati)": p})
null_spatial = pd.DataFrame(rows)
display(null_spatial.round(4)); null_spatial.to_csv(OUT_DIR / "D_null_spatial.csv", index=False, encoding="utf-8-sig")
RESULTS["null_spasial"] = null_spatial.round(5).to_dict("records")
print("Catatan: untuk spat_ratio, nilai kecil = kompak; kolom p di sana tidak bermakna. Bandingkan 'teramati' dengan 'null_rata2'.")
print("Catatan: null Voronoi tidak menjamin kontiguitas polygon atau ukuran komunitas seimbang.")
print("P satu sisi untuk spat_ratio tidak ditafsirkan sebagai bukti signifikansi karena nilai lebih kecil berarti lebih kompak.")


In [ ]:
# Simpan seluruh hasil simulasi null
null_df.to_csv(
    OUT_DIR / "D_null_spatial_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

# Hitung lower-tail p-value untuk spatial ratio
p_lower = (
    1 + (null_df["spat_ratio"] <= obs["spat_ratio"]).sum()
) / (len(null_df) + 1)

print("Jumlah simulasi:", len(null_df))
print("Observed spatial ratio:", obs["spat_ratio"])
print("Lower-tail p-value:", p_lower)

## E. Bootstrap tiket di dalam node (RQ2)

Untuk tiap replikasi: tiket di **setiap node** diambil ulang dengan pengembalian (jumlah sama), embedding node dihitung ulang dari embedding tiket (pipeline Notebook 03; mean global untuk centering **dihitung ulang dari tiket hasil resampling pada setiap replikasi**), graf dibangun ulang dengan **konfigurasi tetap** (kandidat, M), lalu Louvain dijalankan (terbaik dari 10 seed berdasarkan modularity). Ini mengukur ketidakpastian akibat sampel tiket, termasuk pada node berpengaduan sedikit.

Keluaran: distribusi ARI terhadap partisi utama, serta **stabilitas per node** = rata-rata proporsi anggota komunitas teramati yang tetap sekomunitas di seluruh replikasi (1 = inti stabil, rendah = perbatasan).

In [ ]:
rng = np.random.default_rng(SEED)
order = np.argsort(nid, kind="stable")
raw_sorted = raw[order]
n_per = np.bincount(nid, minlength=N)
starts = np.concatenate([[0], np.cumsum(n_per)[:-1]])
node_of_sorted = np.repeat(np.arange(N), n_per)

ari_boot, ncomm_boot = [], []
coassign = np.zeros((N, N), dtype=np.int32)
for b in range(B_BOOT):
    # Resampling embedding tiket mentah; mean global dihitung ulang per replikasi.
    idx = starts[node_of_sorted] + np.floor(rng.random(len(node_of_sorted)) * n_per[node_of_sorted]).astype(int)
    raw_b = raw_sorted[idx]
    raw_b_centered = l2(raw_b - raw_b.mean(0))
    S_b = np.add.reduceat(raw_b_centered, starts, axis=0)
    Xb = l2(S_b / n_per[:, None]).astype(np.float64)
    Wb = graph_from_similarity(Xb @ Xb.T, A_main, cfg["m"])
    best = max((louvain_labels(Wb, 1000 * b + j) for j in range(len(SEEDS))),
               key=lambda t: nx.community.modularity(t[1], t[2], weight="weight"))
    lab_b = best[0]
    ari_boot.append(ARI(lab_main, lab_b)); ncomm_boot.append(len(np.unique(lab_b)))
    coassign += (lab_b[:, None] == lab_b[None, :])
coassign = coassign / B_BOOT

same_main = (lab_main[:, None] == lab_main[None, :]) & ~np.eye(N, dtype=bool)
node_stab = np.array([coassign[i, same_main[i]].mean() if same_main[i].any() else np.nan for i in range(N)])
nodes["stabilitas_bootstrap"] = node_stab
ci = np.percentile(ari_boot, [2.5, 97.5])
print(f"Bootstrap (B = {B_BOOT}): ARI vs partisi utama rata-rata {np.mean(ari_boot):.3f} (95% interval {ci[0]:.3f}-{ci[1]:.3f}) | jumlah komunitas {min(ncomm_boot)}-{max(ncomm_boot)}")
print("Co-assignment dalam komunitas teramati:", round(float(coassign[same_main].mean()), 3),
      "| antar komunitas:", round(float(coassign[~same_main & ~np.eye(N, dtype=bool)].mean()), 3))
rho = spearmanr(node_stab, np.log(nodes["n_pengaduan"]), nan_policy="omit")[0]
print(f"Korelasi stabilitas node dengan log(n_pengaduan): rho = {rho:.3f}")
by_comm = nodes.groupby("komunitas").agg(n_node=("node_id", "size"), stabilitas_rata2=("stabilitas_bootstrap", "mean"),
                                         stabilitas_min=("stabilitas_bootstrap", "min")).round(3)
display(by_comm)
low = nodes.loc[nodes.n_pengaduan < 10, ["kelurahan", "kecamatan", "tahun", "n_pengaduan", "komunitas", "stabilitas_bootstrap"]]
print("Node berpengaduan < 10:"); display(low.round(3))
nodes[["node_id", "lokasi_key", "kelurahan", "kecamatan", "tahun", "n_pengaduan", "komunitas", "stabilitas_bootstrap"]].to_csv(
    OUT_DIR / "E_node_bootstrap_stability.csv", index=False, encoding="utf-8-sig")
RESULTS["bootstrap"] = {"B": B_BOOT, "ARI_mean": float(np.mean(ari_boot)), "ARI_CI95": [float(ci[0]), float(ci[1])],
                        "n_komunitas_rentang": [int(min(ncomm_boot)), int(max(ncomm_boot))],
                        "rho_stabilitas_vs_logn": float(rho), "node_stabilitas_<0.5": int((node_stab < 0.5).sum()),
                        "centering": "mean global dihitung ulang pada setiap replikasi"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(ari_boot, bins=15); axes[0].set_xlabel("ARI (bootstrap vs partisi utama)"); axes[0].set_ylabel("replikasi")
axes[1].scatter(nodes.n_pengaduan, node_stab, s=14, alpha=0.7); axes[1].set_xscale("log")
axes[1].set_xlabel("n_pengaduan per node (log)"); axes[1].set_ylabel("stabilitas bootstrap node")
plt.tight_layout(); plt.savefig(OUT_DIR / "E_bootstrap.png", dpi=150); plt.show()

## F. Evaluasi label topik otomatis

In [ ]:
rng = np.random.default_rng(SEED)
txt = df["teks_embedding"].fillna("").astype(str)
ok = txt.str.strip().ne("") & ~txt.str.contains(r"(?:\d[\s.\-]?){9,}", regex=True)      # buang tiket yang masih memuat deret >= 9 digit
pick = []
for t_, g_ in df[ok].groupby("kelompok_topik"):
    pick += g_.sample(n=min(N_PER_TOPIC, len(g_)), random_state=SEED).index.tolist()
pick = list(rng.permutation(pick))
sheet = pd.DataFrame({"sample_id": np.arange(1, len(pick) + 1), "teks_masked": txt.loc[pick].str.slice(0, 600).to_numpy()})
key = sheet.assign(label_otomatis=df.loc[pick, "kelompok_topik"].to_numpy(), tahun=df.loc[pick, "tahun"].to_numpy(),
                   share_kelas_populasi=df.loc[pick, "kelompok_topik"].map(df["kelompok_topik"].value_counts(normalize=True)).to_numpy())
annot = sheet.assign(label_manual="")
annot.to_csv(OUT_DIR / "F_label_sheet_annotator.csv", index=False, encoding="utf-8-sig")
key.drop(columns=["teks_masked"]).to_csv(OUT_DIR / "F_label_key_JANGAN_DIBAGIKAN.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"opsi_label": topics + ["TIDAK_JELAS"]}).to_csv(OUT_DIR / "F_label_options.csv", index=False, encoding="utf-8-sig")
print(f"Lembar sampel: {len(sheet)} tiket ({N_PER_TOPIC} per topik, bila tersedia). Berkas di:", OUT_DIR)
# Subset 100 baris acak untuk isian manusia tanpa melihat label (seed terpisah agar lembar utama tidak berubah)
rng_blind = np.random.default_rng(SEED + 7)
blind_ids = np.sort(rng_blind.choice(sheet["sample_id"].to_numpy(), size=min(100, len(sheet)), replace=False))
annot[annot["sample_id"].isin(blind_ids)].to_csv(OUT_DIR / "F_label_blind100_template.csv", index=False, encoding="utf-8-sig")
print(f"Template isian buta: {len(blind_ids)} baris -> F_label_blind100_template.csv")
display(annot.head(3))

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
f_ref = OUT_DIR / "F_label_sheet_annotator1.csv"
blind_files = sorted(OUT_DIR.glob("F_label_blind_*.csv"))
valid_opts = set(topics) | {"TIDAK_JELAS"}

if not f_ref.exists():
    print("Belum ada label referensi. Salin F_label_sheet_annotator.csv menjadi F_label_sheet_annotator1.csv, isi label_manual, lalu jalankan ulang sel ini.")
else:
    ref = pd.read_csv(f_ref)[["sample_id", "label_manual"]].rename(columns={"label_manual": "ref"})
    ev = key.merge(ref, on="sample_id", how="left")
    assert ev["ref"].notna().all(), "Label referensi belum terisi penuh (ada baris kosong)."
    assert ev["ref"].isin(valid_opts).all(), "Ada label referensi di luar daftar opsi."
    coverage = float((ev["ref"] != "TIDAK_JELAS").mean())
    e = ev[ev["ref"] != "TIDAK_JELAS"].copy()
    labels_eval = sorted(set(e["label_otomatis"]) | set(e["ref"]))
    cm = pd.crosstab(pd.Categorical(e["label_otomatis"], categories=labels_eval),
                     pd.Categorical(e["ref"], categories=labels_eval),
                     rownames=["label_otomatis"], colnames=["label_referensi"], dropna=False)
    sample_n = e["label_otomatis"].value_counts()
    pop_share = df["kelompok_topik"].value_counts(normalize=True)
    w = e["label_otomatis"].map(lambda z: pop_share[z] / sample_n[z]).to_numpy(float)
    w = w / w.sum()
    correct = (e["ref"].to_numpy() == e["label_otomatis"].to_numpy()).astype(float)
    acc_w = float(np.sum(w * correct))
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        e["label_otomatis"], e["ref"], labels=labels_eval, average="weighted",
        sample_weight=w, zero_division=0)
    rng_b = np.random.default_rng(SEED)
    boots = []
    groups_eval = {cl: g_ for cl, g_ in e.groupby("label_otomatis")}
    for _ in range(2000):
        acc_by_class = {}
        for cl, g_ in groups_eval.items():
            vals = (g_["ref"].to_numpy() == g_["label_otomatis"].to_numpy()).astype(float)
            acc_by_class[cl] = float(rng_b.choice(vals, len(vals), replace=True).mean())
        boots.append(sum(float(pop_share[cl]) * acc_by_class[cl] for cl in groups_eval) /
                     sum(float(pop_share[cl]) for cl in groups_eval))
    ci_acc = np.percentile(boots, [2.5, 97.5])
    kappa_auto = cohen_kappa_score(e["ref"], e["label_otomatis"])
    print(f"Cakupan label referensi: {coverage:.1%}; TIDAK_JELAS: {1-coverage:.1%}")
    print(f"Akurasi berbobot populasi terhadap label referensi: {acc_w:.3f} (95% bootstrap CI {ci_acc[0]:.3f}-{ci_acc[1]:.3f})")
    print(f"Precision/recall/F1 weighted: {p_w:.3f}/{r_w:.3f}/{f_w:.3f}")
    print(f"Kappa referensi vs otomatis: {kappa_auto:.3f} (kesesuaian dengan label referensi, BUKAN reliabilitas antar-pelabel)")
    res_f = {"n_dinilai": int(len(e)), "cakupan_non_tidak_jelas": coverage, "tidak_jelas": 1 - coverage,
             "akurasi_berbobot": acc_w, "precision_weighted": float(p_w), "recall_weighted": float(r_w),
             "f1_weighted": float(f_w), "CI95_akurasi": [float(ci_acc[0]), float(ci_acc[1])],
             "kappa_referensi_vs_otomatis": float(kappa_auto)}

    ev_i = ev.set_index("sample_id")
    blind = {}
    for f_ in blind_files:
        nm = f_.stem.replace("F_label_blind_", "")
        b_ = pd.read_csv(f_)[["sample_id", "label_manual"]]
        b_ = b_[b_["label_manual"].notna() & b_["label_manual"].astype(str).str.strip().ne("")]
        assert b_["label_manual"].isin(valid_opts).all(), f"Ada label di luar daftar opsi pada {f_.name}"
        blind[nm] = b_.drop_duplicates("sample_id").set_index("sample_id")["label_manual"]
    res_blind, pair_rows = {}, []
    for nm, s_ in blind.items():
        d_ = ev_i.loc[ev_i.index.intersection(s_.index)].assign(blind=s_)
        d_ = d_[(d_["blind"] != "TIDAK_JELAS") & (d_["ref"] != "TIDAK_JELAS")]
        if len(d_) < 2:
            continue
        res_blind[nm] = {"n": int(len(d_)),
                         "kappa_vs_referensi": float(cohen_kappa_score(d_["blind"], d_["ref"])),
                         "kappa_vs_otomatis": float(cohen_kappa_score(d_["blind"], d_["label_otomatis"])),
                         "akurasi_vs_referensi": float((d_["blind"] == d_["ref"]).mean())}
    nms = list(blind)
    for i_ in range(len(nms)):
        for j_ in range(i_ + 1, len(nms)):
            common = blind[nms[i_]].index.intersection(blind[nms[j_]].index)
            a_, b2_ = blind[nms[i_]].loc[common], blind[nms[j_]].loc[common]
            m_ = (a_ != "TIDAK_JELAS") & (b2_ != "TIDAK_JELAS")
            if m_.sum() >= 2:
                pair_rows.append({"pelabel_1": nms[i_], "pelabel_2": nms[j_], "n": int(m_.sum()),
                                  "kappa": float(cohen_kappa_score(a_[m_], b2_[m_]))})
    if res_blind:
        print("\nIsian manusia tanpa melihat label (boleh dilaporkan sebagai kappa manusia):")
        display(pd.DataFrame(res_blind).T.round(3))
        res_f["isian_buta"] = res_blind
    else:
        print("\nBelum ada isian manusia tanpa melihat label (F_label_blind_<nama>.csv). "
              "Jangan menulis kappa antar-pelabel atau 'validasi manual' di naskah.")
    if pair_rows:
        print("Kappa antar-pelabel manusia:"); display(pd.DataFrame(pair_rows).round(3))
        res_f["kappa_antar_pelabel"] = pair_rows
    display(cm)
    cm.to_csv(OUT_DIR / "F_confusion_matrix.csv", encoding="utf-8-sig")
    RESULTS["validasi_manual"] = res_f

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

f_ref_b1 = OUT_DIR / "F_label_sheet_annotator1.csv"
assert f_ref_b1.exists() and "pick" in globals(), "Jalankan sel pembuatan lembar sampel (bagian F) dan letakkan label referensi di folder keluaran lebih dulu."
ref_b1 = pd.read_csv(f_ref_b1)[["sample_id", "label_manual"]]
pos_all = df.index.get_indexer(pick)                      
ref_b1 = ref_b1.assign(pos=pos_all[ref_b1["sample_id"].to_numpy() - 1])
ref_b1 = ref_b1[ref_b1["label_manual"] != "TIDAK_JELAS"]
test_pos = ref_b1["pos"].to_numpy(); y_test = ref_b1["label_manual"].to_numpy()

rng_c = np.random.default_rng(SEED)
train_pool = np.setdiff1d(np.arange(len(df)), pos_all)
train_pos = rng_c.choice(train_pool, size=min(PROBE_N, len(train_pool)), replace=False)
y_train = df["kelompok_topik"].to_numpy()[train_pos]

feats = {"IndoBERT (mean_centered)": l2(raw - raw.mean(0)), "Sentence-transformer": Ec_sb}
names = list(feats); pred = {}
for nm_, X_ in feats.items():
    clf = LogisticRegression(max_iter=500, C=1.0).fit(X_[train_pos], y_train)
    pred[nm_] = clf.predict(X_[test_pos])
tab_b1 = pd.DataFrame([{
    "representasi": nm_, "n_uji": int(len(y_test)),
    "akurasi_vs_label_referensi": round(float(accuracy_score(y_test, pred[nm_])), 4),
    "macro_F1": round(float(f1_score(y_test, pred[nm_], average="macro")), 4)} for nm_ in names])
print("Klasifikasi: dilatih pada weak labels, diuji pada label referensi (bukan aturan kata kunci):"); display(tab_b1)

rng_b2 = np.random.default_rng(SEED)
c_a = (pred[names[0]] == y_test).astype(float); c_b = (pred[names[1]] == y_test).astype(float)
idxs = [rng_b2.integers(0, len(y_test), len(y_test)) for _ in range(5000)]
d_ci = np.percentile([c_b[i].mean() - c_a[i].mean() for i in idxs], [2.5, 97.5])
d_obs = float(c_b.mean() - c_a.mean())
print(f"Selisih akurasi (sentence-transformer - IndoBERT) = {d_obs:+.3f} (95% bootstrap CI {d_ci[0]:+.3f} s.d. {d_ci[1]:+.3f})")

W_sb = EXTRA_GRAPHS[list(EXTRA_GRAPHS)[0]]
best_sb = None
for res_ in RES_GRID:
    for s_ in SEEDS:
        lab_, g_, p_ = louvain_labels(W_sb, s_, res_)
        if len(np.unique(lab_)) == K_MAIN:
            q_ = nx.community.modularity(g_, p_, weight="weight")
            if best_sb is None or q_ > best_sb[0]:
                best_sb = (q_, lab_, res_, s_)
res_b1 = {"akurasi": tab_b1.set_index("representasi")["akurasi_vs_label_referensi"].to_dict(),
          "selisih_akurasi_sbert_minus_indobert": d_obs, "CI95_selisih": [float(d_ci[0]), float(d_ci[1])]}
if best_sb is None:
    print(f"Tidak ada partisi sentence-transformer dengan tepat {K_MAIN} komunitas pada RES_GRID.")
else:
    ari_sb, nmi_sb = float(ARI(lab_main, best_sb[1])), float(NMI(lab_main, best_sb[1]))
    print(f"Partisi IndoBERT (utama) vs sentence-transformer pada K={K_MAIN}: ARI = {ari_sb:.3f} | NMI = {nmi_sb:.3f} (resolusi {best_sb[2]}, seed {best_sb[3]})")
    display(pd.crosstab(pd.Series(lab_main, name="komunitas_IndoBERT"), pd.Series(best_sb[1], name="komunitas_SBERT")))
    res_b1.update({"ARI_partisi_indobert_vs_sbert": ari_sb, "NMI_partisi_indobert_vs_sbert": nmi_sb})
RESULTS["representasi_label_referensi"] = res_b1

In [ ]:
from sklearn.metrics import f1_score
f1_macro = f1_score(e["ref"], e["label_otomatis"], average="macro",
                    labels=labels_eval, zero_division=0)
print(f"Macro-F1: {f1_macro:.3f}")

## H. Analisis temporal: partisi per tahun

Partisi utama memakai edge lintas tahun, sehingga kesinambungan komunitas antar-tahun sebagian merupakan akibat desain graf. Bagian ini membentuk partisi **terpisah per tahun** (graf sama, kandidat edge hanya dalam tahun yang sama), lalu membandingkan: 
- (1) partisi per tahun vs partisi utama; 
- (2) partisi antar-tahun pada kelurahan yang sama (ARI, NMI, pencocokan optimal) terhadap **null yang mengacak label di dalam kecamatan**; 
- (3) komposisi node dan pengaduan per komunitas per tahun. Klaim dinamika atau kesinambungan temporal hanya boleh ditulis bila hasil 
- (4) mendukung; ARI antar-seed dalam tahun yang sama menjadi batas atas stabilitas yang bisa diharapkan.

In [ ]:
# Analisis temporal: partisi per tahun (tanpa edge lintas tahun) lalu dibandingkan antar-tahun
from scipy.optimize import linear_sum_assignment

YEARS = [int(y_) for y_ in sorted(np.unique(yr))]
B_TEMPORAL_NULL = 999

# Graf yang sama dengan graf utama (bobot IndoBERT, Kgeo, M) tetapi kandidat edge hanya dalam tahun yang sama
A_same = candidate_matrix("same_year", cfg["k_geo"])
W_same = graph_from_similarity(SIM, A_same, cfg["m"])

per_year = {}
for y_ in YEARS:
    idx_ = np.flatnonzero(yr == y_)
    Wy = W_same[np.ix_(idx_, idx_)]
    labs_, qs_ = [], []
    for s_ in SEEDS:
        lab_, g_, p_ = louvain_labels(Wy, s_)
        labs_.append(lab_); qs_.append(nx.community.modularity(g_, p_, weight="weight"))
    best_ = int(np.argmax(qs_))
    ari_seed_ = float(np.mean([ARI(labs_[i], labs_[j]) for i in range(len(SEEDS)) for j in range(i + 1, len(SEEDS))]))
    per_year[y_] = dict(idx=idx_, lab=labs_[best_], q=float(qs_[best_]),
                        k=int(len(np.unique(labs_[best_]))), ari_seed=ari_seed_)

tab_year = pd.DataFrame([{
    "tahun": y_, "n_node": len(d_["idx"]), "n_komunitas": d_["k"], "modularity": round(d_["q"], 4),
    "ARI_antar_seed": round(d_["ari_seed"], 4),
    "ARI_vs_partisi_utama": round(float(ARI(lab_main[d_["idx"]], d_["lab"])), 4),
    "NMI_vs_partisi_utama": round(float(NMI(lab_main[d_["idx"]], d_["lab"])), 4),
} for y_, d_ in per_year.items()])
print("Partisi per tahun (graf tanpa edge lintas tahun):"); display(tab_year)

# Label per kelurahan pada tiap tahun
kec_by_kel = nodes.drop_duplicates("lokasi_key").set_index("lokasi_key")["kecamatan"]
lab_kel = {y_: pd.Series(d_["lab"], index=lk[d_["idx"]]) for y_, d_ in per_year.items()}

def matched_share(a, b):
    ct_ = pd.crosstab(a, b).to_numpy()
    r_, c_ = linear_sum_assignment(-ct_)
    return float(ct_[r_, c_].sum() / ct_.sum())

rng_t = np.random.default_rng(SEED)
pair_rows = []
for i_, y1 in enumerate(YEARS):
    for y2 in YEARS[i_ + 1:]:
        shared = sorted(set(lab_kel[y1].index) & set(lab_kel[y2].index))
        a = lab_kel[y1].loc[shared].to_numpy(); b = lab_kel[y2].loc[shared].to_numpy()
        obs_ari = float(ARI(a, b))
        # Null: label tahun kedua diacak antar kelurahan DALAM kecamatan yang sama
        # (mempertahankan pola geografis kasar tingkat kecamatan)
        kec_ = kec_by_kel.loc[shared].to_numpy()
        groups_ = [np.flatnonzero(kec_ == k_) for k_ in np.unique(kec_)]
        null_ = np.empty(B_TEMPORAL_NULL)
        for t_ in range(B_TEMPORAL_NULL):
            bp = b.copy()
            for g_idx in groups_:
                bp[g_idx] = b[rng_t.permutation(g_idx)]
            null_[t_] = ARI(a, bp)
        pair_rows.append({
            "tahun_1": y1, "tahun_2": y2, "n_kelurahan_bersama": len(shared),
            "ARI": round(obs_ari, 4), "NMI": round(float(NMI(a, b)), 4),
            "kelurahan_tetap_setelah_pencocokan": round(matched_share(a, b), 4),
            "ARI_null_dalam_kecamatan_rata2": round(float(null_.mean()), 4),
            "p_vs_null_kecamatan": round(float((1 + (null_ >= obs_ari).sum()) / (B_TEMPORAL_NULL + 1)), 4),
        })
tab_pairs = pd.DataFrame(pair_rows)
print("Kesesuaian partisi antar-tahun (kelurahan yang sama):"); display(tab_pairs)
print("Pembanding batas atas: ARI antar-seed dalam tahun yang sama =",
      tab_year["ARI_antar_seed"].round(3).tolist())

# Komposisi node partisi utama per tahun (jangan menafsirkan perubahan volume tanpa tabel ini)
comp_nodes = pd.crosstab(pd.Series(lab_main, name="komunitas"), pd.Series(yr, name="tahun"))
vol = dfm.groupby(["komunitas", "tahun"]).size().unstack(fill_value=0)
comp = comp_nodes.join(vol, lsuffix="_node", rsuffix="_pengaduan")
print("Jumlah node dan pengaduan per komunitas per tahun (partisi utama):"); display(comp)
print("Total pengaduan kota per tahun:", dfm.groupby("tahun").size().to_dict())

tab_year.to_csv(OUT_DIR / "H_per_year_partition.csv", index=False, encoding="utf-8-sig")
tab_pairs.to_csv(OUT_DIR / "H_temporal_pairs.csv", index=False, encoding="utf-8-sig")
comp.to_csv(OUT_DIR / "H_node_composition_main.csv", encoding="utf-8-sig")
lab_year_full = np.zeros(N, int)
for y_, d_ in per_year.items():
    lab_year_full[d_["idx"]] = d_["lab"]
pd.DataFrame({"lokasi_key": lk, "tahun": yr, "komunitas_utama": lab_main,
              "komunitas_per_tahun": lab_year_full}
             ).to_csv(OUT_DIR / "H_membership_per_year.csv", index=False, encoding="utf-8-sig")

RESULTS["temporal"] = {"per_tahun": tab_year.to_dict("records"), "antar_tahun": tab_pairs.to_dict("records")}

In [ ]:
json.dump(RESULTS, open(OUT_DIR / "06_key_results.json", "w"), indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Tersimpan:", OUT_DIR / "06_key_results.json")
print("Bagian hasil yang terisi:", list(RESULTS))
if "validasi_manual" not in RESULTS:
    print("Perhatian: validasi manual (bagian F) belum dijalankan; klaim tentang validitas label topik belum boleh ditulis.")
if "temporal" not in RESULTS:
    print("Perhatian: analisis temporal (bagian H) belum dijalankan; klaim dinamika temporal belum boleh ditulis.")
if "representasi_sbert" not in RESULTS:
    print("Perhatian: pembanding sentence-transformer (B0) belum dijalankan; klaim keunggulan IndoBERT belum boleh ditulis.")
if "validasi_manual" in RESULTS and "isian_buta" not in RESULTS["validasi_manual"]:
    print("Catatan: belum ada isian manusia tanpa melihat label; tulis bagian F sebagai kesesuaian dengan label referensi (pemeriksaan peneliti).")

In [ ]:
# Simpan seluruh hasil simulasi null
null_df.to_csv(
    OUT_DIR / "D_null_spatial_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

# Hitung p-value lower-tail untuk spat_ratio
p_lower = (
    1 + (null_df["spat_ratio"] <= obs["spat_ratio"]).sum()
) / (len(null_df) + 1)

print("Lower-tail p-value:", p_lower)

In [ ]:
p_compact = (1 + (null_df["spat_ratio"] <= obs["spat_ratio"]).sum()) / (len(null_df) + 1)

In [ ]:
purity_komunitas = ct.max(axis=0).sum() / ct.values.sum()

In [ ]:
nm = "Louvain, bobot IndoBERT (graf utama)"
sub = per_run[(per_run.k_target == K_MAIN) & (per_run.metode == nm)]
print(len(sub))
print(sub[["gain_indobert", "gain_lsa", "gain_topik", "spat_ratio"]].round(4).to_string())
print("mean:", sub[["gain_indobert", "gain_lsa", "gain_topik", "spat_ratio"]].mean().round(4).to_dict())
print("sd:  ", sub[["gain_indobert", "gain_lsa", "gain_topik", "spat_ratio"]].std().round(4).to_dict())
print("partisi final:", ref.iloc[0][["gain_indobert", "gain_lsa", "gain_topik", "spat_ratio"]].round(4).to_dict())

In [ ]:
print("Kolom ref:", ref.columns.tolist())
print("Isi ref:")
display(ref.head())
print("Bentuk ref:", ref.shape)

In [ ]:
# Cari variabel dataframe yang memiliki kolom metrik
for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)
        metrics = {
            "gain_indobert", "gain_lsa",
            "gain_topik", "spat_ratio"
        }
        if metrics.issubset(cols):
            print(name, obj.shape, obj.columns.tolist())

In [ ]:
from sklearn.metrics import adjusted_rand_score as ARI
nm = "Louvain, bobot IndoBERT (graf utama)"
labs = [l for m, k, l in runs if m == nm and k == K_MAIN]
dup = [(i, j) for i in range(len(labs)) for j in range(i+1, len(labs)) if ARI(labs[i], labs[j]) == 1.0]
print(len(labs), dup)